In [1]:
pip install feedparser openai schedule python-dotenv


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import time
import feedparser
import schedule
import smtplib
from dotenv import load_dotenv
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from openai import OpenAI

In [3]:
# LOAD ENV VARIABLES

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
EMAIL_USER = os.getenv("EMAIL_USER")
EMAIL_PASS = os.getenv("EMAIL_PASS")
EMAIL_TO   = os.getenv("EMAIL_TO")

client = OpenAI(api_key=OPENAI_API_KEY)

In [4]:
# RSS FEEDS

RSS_FEEDS = [
    "https://rss.cnn.com/rss/edition.rss",
    "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml",
    "https://feeds.bbci.co.uk/news/rss.xml",
    "https://www.aljazeera.com/xml/rss/all.xml",
    "https://www.theguardian.com/world/rss",
    "https://techcrunch.com/feed/",
    "https://www.reddit.com/r/worldnews/.rss",
]

In [ ]:
# STEP 1: FETCH NEWS

def fetch_rss_feeds(feed_urls, limit=50):
    articles = []
    for url in feed_urls:
        feed = feedparser.parse(url)
        for entry in feed.entries[:limit]:
            articles.append({
                "title": entry.title,
                "link": entry.link,
                "summary": entry.get("summary", "")
            })
    return articles

In [21]:
# STEP 2: SUMMARIZE USING OPENAI

def summarize_text(text):
    text = (text or "").strip()
    if not text:
        return "No article summary available."

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": f"Summarize this news article in 10–12 sentences:\n{text}"
                }
            ],
            max_tokens=120
        )
        return response.choices[0].message.content.strip()
    except Exception:
        return text[:750] + ("..." if len(text) > 750 else "")


In [7]:
# STEP 3: BUILD HTML EMAIL

def build_html_email(news_items):
    html = """
    <html>
    <body>
        <h1>📰 AI-POWERED DAILY NEWS AGGREGATOR</h1>
        <hr>
    """
    for item in news_items:
        html += f"""
        <h2><a href="{item['link']}">{item['headline']}</a></h2>
        <p>{item['summary']}</p>
        <hr>
        """
    html += "</body></html>"
    return html

In [8]:
# STEP 4: SEND EMAIL

def send_email(html_content):
    msg = MIMEMultipart("alternative")
    msg["Subject"] = "AI-Powered Daily News Aggregator"
    msg["From"] = EMAIL_USER
    msg["To"] = EMAIL_TO

    msg.attach(MIMEText(html_content, "html"))

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(EMAIL_USER, EMAIL_PASS)
        server.send_message(msg)


In [9]:
# SEND EMAIL TO MULTIPLE RECIPIENTS
def send_email(html_content):
    recipients = [email.strip() for email in EMAIL_TO.split(",")]

    msg = MIMEMultipart("alternative")
    msg["Subject"] = "AI-POWERED Daily News Digest"
    msg["From"] = EMAIL_USER
    msg["To"] = ", ".join(recipients)

    msg.attach(MIMEText(html_content, "html"))

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(EMAIL_USER, EMAIL_PASS)
        server.send_message(msg, from_addr=EMAIL_USER, to_addrs=recipients)


In [20]:
# STEP 5: FULL PIPELINE JOB

def daily_news_job():
    print("🚀 Running Daily News Pipeline...")

    articles = fetch_rss_feeds(RSS_FEEDS, limit=12)

    summarized_news = []
    for article in articles[:12]:  # Keep the test run light and avoid API bursts
        summary = summarize_text(article["summary"] or article["title"])
        summarized_news.append({
            "headline": article["title"],
            "link": article["link"],
            "summary": summary
        })

    email_html = build_html_email(summarized_news)
    send_email(email_html)

    print("✅ Daily News Email Sent Successfully")


In [22]:
# STEP 6: SCHEDULER

daily_news_job()
# schedule.every().day.at("08:00").do(daily_news_job)

# print("⏰ Scheduler started...")

# while True:
#     schedule.run_pending()
#     time.sleep(30)

print("✅ Job completed. Exiting program.")
print("✅ Test run finished. Exiting.")

🚀 Running Daily News Pipeline...
✅ Daily News Email Sent Successfully
✅ Job completed. Exiting program.
✅ Test run finished. Exiting.
